# Gestion de la Mémoire et Garbage Collector en Python

Dans ce notebook, nous allons voir :

- comment Python gère la **mémoire** pour les objets,
- ce qu’est le **compteur de références**,
- comment fonctionne le **garbage collector** (`gc`),
- comment éviter les **fuites mémoires** (cycles, références persistantes),
- comment cela impacte des **workflows de data science** (boucles d’entraînement, gros tableaux, etc.).

L’objectif : comprendre ce qui se passe “sous le capot” pour écrire du code plus robuste et plus efficace.

# 1. Rappels : objets, références et mémoire en Python

En Python :

- **tout est objet**, stocké sur le **tas** (heap) c'est la zone mémoire où Python stocke **tous les objets**,
- les variables sont juste des **références** vers ces objets,
- Python gère la mémoire **automatiquement** (pas de `free()` manuel comme en `C`),
- la libération se fait principalement via un **compteur de références**, complété par un **garbage collector**. Chaque objet sait combien de variables ou d’autres objets pointent vers lui. Quand ce compteur atteint **0**, l’objet est automatiquement détruit. Et un **garbage collector** supplémentaire sert uniquement à repérer les **cycles** (ex : une liste qui se contient elle-même)

Idée clé :  
> Un objet est détruit dès qu’il n’est plus référencé nulle part (sauf cas de cycles).

## Variables et références

On va utiliser la fonction `id(objet)` qui retourne **l’adresse mémoire** (ou un identifiant unique) de l’objet stocké sur le heap.

- Si deux variables pointent vers **le même objet**, elles ont **le même `id`**.
- Si elles pointent vers **deux objets différents**, même si leur contenu est identique, leurs `id` seront **différents**.

In [1]:
a = [1, 2, 3]
b = a    
c = [1, 2, 3]

In [2]:
print("id(a) :", id(a))
print("id(b) :", id(b))
print("id(c) :", id(c))

id(a) : 4496942144
id(b) : 4496942144
id(c) : 4497017984


Ici, `b` pointe vers le même objet que `a` et `c` pointe vers un autre objet

# 2. Compteur de références

Python associe à chaque objet un **compteur de références** : chaque fois qu’une variable, une liste ou un autre objet pointe vers lui alors le compteur augmente; chaque fois qu’une référence disparaît alors il diminue. Quand il atteint **0**, l’objet est détruit.

`sys.getrefcount(x)` permet d’inspecter ce compteur.

⚠️ Attention : **l’appel à `getrefcount(x)` ajoute lui-même une référence temporaire**, car `x` est passé en argument à la fonction.  
Le compteur affiché est donc **+1** par rapport au nombre réel de références actives.

In [1]:
import sys

On initialise une variable `x` qui est une liste, et on affiche son compteur de références avec `sys.getrefcount(x)`.

In [3]:
x = [1, 2, 3]
print("Références sur x :", sys.getrefcount(x))  # inclut la référence passée à getrefcount

Références sur x : 2


On crée une nouvelle référence `y` vers le même objet que `x`, et on affiche à nouveau le compteur de références.

In [5]:
y = x
print("Après y = x :", sys.getrefcount(x))

Après y = x : 5


On crée un objet `z` qui contient 2 références vers le même objet que `x`, et on affiche à nouveau le compteur de références.

In [7]:
z = [x, x]
print("Après z = [x, x] :", sys.getrefcount(x))

Après z = [x, x] : 5


In [8]:
del y
print("Après del y :", sys.getrefcount(x))

Après del y : 4


In [9]:
del z
print("Après del z :", sys.getrefcount(x))

Après del z : 2


# 3. Quand un objet est-il libéré ?

Pour tous les objets **sans cycle**, Python les détruit automatiquement dès que leur **compteur de références tombe à 0**. Cela signifie qu’il n’existe plus **aucune variable** ou **aucune structure** pointant vers eux.

Dans l’exemple :

In [12]:
def creer_grande_liste():
    # liste avec 10 millions de zéros
    data = [0] * 10_000_000
    print("Références dans la fonction :", sys.getrefcount(data))
    return data

In [13]:
lst = creer_grande_liste()
print("Références après retour :", sys.getrefcount(lst))

Références dans la fonction : 2
Références après retour : 2


À l’intérieur de la fonction, l’objet `data` a une référence (plus celle due à `getrefcount`).

Quand la fonction se termine, la variable locale `data` disparaît, mais l’objet n’est **pas détruit**, car il est renvoyé et désormais référencé par `lst`.

Si on supprime `lst` :

In [14]:
del lst

- il n’existe plus **aucune** référence vers la grande liste  
- son compteur passe à 0  
- Python peut libérer la mémoire immédiatement.

> Idée clé : un objet survit tant qu’au moins **une** référence pointe vers lui. Dès que ce n’est plus le cas, il devient libérable.

# 4. Pourquoi les cycles posent problème ?

Le compteur de références fonctionne bien… sauf quand **des objets se référencent entre eux**.  
Exemple : `a` pointe vers `b` et `b` pointe vers `a`.  
Même si plus aucune variable du programme ne pointe vers eux, leur compteur ne descend **jamais à 0**, car ils se tiennent mutuellement en vie.

C’est pourquoi Python ajoute un **garbage collector cyclique** : un module qui repère périodiquement ces groupes d’objets inutiles et les libère.

### Création d'un cycle de références

In [16]:
class Node:
    def __init__(self, name):
        self.name = name
        self.child = None

In [17]:
a = Node("A")
b = Node("B")

a.child = b
b.child = a   # cycle A -> B -> A

Même après :

In [18]:
del a
del b

les deux objets continuent d’exister en mémoire à cause du cycle.  

-> Le garbage collector détecte alors qu’ils ne sont plus accessibles depuis le programme et les libère.

# 5. Le module `gc` : le garbage collector de Python

Le garbage collector (`gc`) est la partie de Python qui s’occupe de repérer et nettoyer les **cycles d’objets** que le compteur de références ne peut pas libérer.

Le module `gc` permet simplement de :
- vérifier si le GC est actif,
- déclencher une collecte manuellement (`gc.collect()`),
- désactiver/réactiver la collecte (rarement utile en pratique),
- analyser quels objets ont été collectés (pour débugger des fuites).

Le GC regarde surtout les **objets conteneurs** (listes, dicts, objets classes…) car ce sont eux qui peuvent créer des cycles.

In [13]:
import gc

In [14]:
# Informations de base sur le GC
print("GC activé ?", gc.isenabled())

GC activé ? True


In [15]:
# Lancer une collecte manuelle
collected = gc.collect()
print("Objets collectés lors du gc.collect():", collected)

Objets collectés lors du gc.collect(): 3


Le compteur de références gère 99% des cas, et le GC vient juste ramasser ce que le compteur ne peut pas libérer (les cycles).

## 7. En Data Science : gros objets, boucles et fuites silencieuses

En data science, on manipule :

- de **gros tableaux** (NumPy, pandas),
- des **modèles** avec beaucoup de paramètres,
- des **boucles d’entraînement** longues.

Pièges classiques :

- conserver des références inutiles (listes qui accumulent des résultats, historique complet des batchs),
- garder en mémoire des modèles intermédiaires non utilisés,
- ne pas libérer des objets avant de relancer un nouvel entraînement.

Une bonne pratique :  
> Identifier les objets lourds et s’assurer qu’aucune référence inutile ne les retient.

In [24]:
# Exemple : accumulation involontaire

import numpy as np
import gc

def create_batch():
    return np.random.randn(1000, 1000)  # ~ gros batch

historique = []

for i in range(10):
    batch = create_batch()
    # On simule un "traitement"
    _ = batch.mean()
    # Mauvaise pratique : on stocke tout
    historique.append(batch)

print("Taille de l'historique :", len(historique))

Taille de l'historique : 10


In [25]:
# Si on n'utilise plus historique, on peut le libérer :
del historique
gc.collect()  # forcer la collecte

749

## 8. Exercice : corriger une fuite mémoire logique

On simule une boucle de traitement de batchs qui :

- crée des lots de données,
- les “traite”,
- mais **ne libère jamais** certaines références.

### Code initial

In [25]:
cache = []

def process():
    for i in range(100):
        batch = np.random.randn(1000, 1000)
        resultat = batch.mean()
        cache.append((batch, resultat))
    return cache

**Problème** : la liste cache garde tous les batchs en mémoire.

1. Modifiez le code pour :
   - ne garder que les résultats scalaires,
   - ou bien un sous-échantillon de batchs (ex. 5 premiers).
2. Assurez-vous que les gros tableaux deviennent libérables après usage.
3. Ajoutez un gc.collect() à un endroit logique.

### Correction

In [27]:
import numpy as np
import gc

def process_corrige():
    resultats = []
    for i in range(100):
        batch = np.random.randn(1000, 1000)
        resultat = batch.mean()
        # On stocke uniquement le scalaire, pas le batch complet
        resultats.append(resultat)
        # à la fin de l'itération, batch n'est plus utilisé → libérable
    # Optionnel : forcer une collecte à la fin d'une grosse boucle
    gc.collect()
    return resultats

In [28]:
res = process_corrige()
print("Nombre de résultats :", len(res))

Nombre de résultats : 100


# Au final

La plupart du temps, Python gère bien la mémoire.  
Les problèmes arrivent quand on crée des **références inutiles** ou des **cycles** non maîtrisés.  

Comprendre le GC permet de diagnostiquer et corriger ces situations.